In [1]:
import os, cv2, numpy as np, tensorflow as tf, joblib
from deepface import DeepFace
# === CELL 0: KHÔI PHỤC MÔI TRƯỜNG (CHẠY ĐẦU TIÊN) ===
print("⏳ Đang cài lại thư viện do Colab reset... (đợi ~60s)")

# Cài đồng bộ, để Colab tự quyết định phiên bản numpy tương thích
!pip install -q deepface tensorflow scikit-learn opencv-python joblib tqdm

# Kiểm tra import
try:
    import os, cv2, numpy as np, tensorflow as tf, joblib
    from deepface import DeepFace
    print("✅ Môi trường đã sẵn sàng! Giờ chạy tiếp cell bên dưới.")
except ImportError as e:
    print(f"Vẫn lỗi: {e}")
    print("Bấm Runtime → Restart Session, rồi chạy lại cell này.")

26-05-29 15:16:43 - Directory /root/.deepface has been created
26-05-29 15:16:43 - Directory /root/.deepface/weights has been created
⏳ Đang cài lại thư viện do Colab reset... (đợi ~60s)
✅ Môi trường đã sẵn sàng! Giờ chạy tiếp cell bên dưới.


In [2]:
# Deepface dùng CNN Facenet/VGGFace bên dưới → vẫn đúng yêu cầu CNN+ANN
!pip install deepface scikit-learn opencv-python tqdm joblib -q

import os, numpy as np, cv2, joblib
from deepface import DeepFace
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from tqdm import tqdm

print("✅ Libraries installed. ⚠️ BẤM RESTART RUNTIME NGAY SAU KHI CHẠY CELL NÀY!")

✅ Libraries installed. ⚠️ BẤM RESTART RUNTIME NGAY SAU KHI CHẠY CELL NÀY!


In [3]:
# Ép numpy về bản ổn định tương thích Colab, xóa cache nhị phân cũ
!pip uninstall numpy -y -q
!pip install "numpy<2.0" -q
print("✅ Fix complete")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 

In [ ]:
DATA_PATH = '/content/drive/MyDrive/60 ẢNH'
print("🚀 Pipeline ready!")

🚀 Pipeline ready!


In [ ]:
# === CHẠY CELL NÀY NGAY SAU KHI RESTART ===
import os
import numpy as np
import cv2
import joblib
from deepface import DeepFace
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from tqdm import tqdm

DATA_PATH = '/content/drive/MyDrive/60 ẢNH'
print("✅ Import xong & set đường dẫn. Chạy tiếp Cell 3.")

26-05-28 11:40:21 - Directory /root/.deepface has been created
26-05-28 11:40:21 - Directory /root/.deepface/weights has been created
✅ Import xong & set đường dẫn. Chạy tiếp Cell 3.


In [ ]:
# Lấy danh sách 33 folder (tên folder = tên người)
class_folders = sorted([f for f in os.listdir(DATA_PATH) if os.path.isdir(os.path.join(DATA_PATH, f))])
print(f"📁 Found {len(class_folders)} classes: {class_folders}")

X, y, skipped = [], [], 0

for idx, person_name in enumerate(class_folders):
    person_path = os.path.join(DATA_PATH, person_name)
    img_files = [f for f in os.listdir(person_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print(f"\n⏳ Processing {person_name} ({len(img_files)} images)...")

    for img_file in tqdm(img_files, desc="  Extracting"):
        try:
            img_path = os.path.join(person_path, img_file)
            # DeepFace tự detect, align, crop face & trích vector 512 chiều bằng CNN
            objs = DeepFace.represent(img_path, model_name='Facenet', enforce_detection=True)

            if len(objs) > 0 and "embedding" in objs[0]:
                X.append(objs[0]["embedding"])
                y.append(idx)
            else:
                skipped += 1
        except Exception:
            skipped += 1  # Ảnh không detect được face / lỗi format / filter quá nặng

X, y = np.array(X), np.array(y)
print(f"\n✅ Valid embeddings: {len(X)} | Skipped (lỗi/không có mặt): {skipped}")
print(f"📊 Shape: {X.shape}")

np.save('/content/X.npy', X)
np.save('/content/y.npy', y)

📁 Found 31 classes: ['BUI DANG KHOI', 'DANG NGUYEN PHUONG NGHI', 'HA PHUONG THAO', 'HOANG BAO TRAN', 'HOANG BUI TRA MY', 'LE HUYNH DUC HUY', 'LE MINH TRIET', 'LE THAI BAO', 'LE THI NHU QUYNH', 'LE TRAN QUY ANH', 'LE TRONG DAI', 'MAI HO QUOC TUY', 'NGUYEN BAO HAN', 'NGUYEN DONG HAI', 'NGUYEN HOANG BAO', 'NGUYEN HUU TOAN', 'NGUYEN KHAC LUU VU', 'NGUYEN NGOC KHANH UYEN', 'NGUYEN NGOC KIM TUYET', 'NGUYEN THI THANH HA', 'NGUYEN TRONG MINH', 'NHAN MANH TUAN', 'PHAM DUC THANH CONG', 'PHAM LY BAO LAM', 'PHAM MAI PHUONG', 'THAI TUAN PHAT', 'TRAN GIA HAN', 'TRAN MINH HOANG', 'TRAN NGOC THAO ANH', 'TRAN THE DANG KHOA', 'TRINH THUY HANG']

⏳ Processing BUI DANG KHOI (60 images)...


  Extracting:   0%|          | 0/60 [00:00<?, ?it/s]

26-05-28 11:40:30 - 🔗 facenet_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facenet_weights.h5 to /root/.deepface/weights/facenet_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facenet_weights.h5
To: /root/.deepface/weights/facenet_weights.h5

  0%|          | 0.00/92.2M [00:00<?, ?B/s]
 12%|█▏        | 11.0M/92.2M [00:00<00:00, 81.9MB/s]
 31%|███       | 28.3M/92.2M [00:00<00:00, 128MB/s] 
 46%|████▌     | 42.5M/92.2M [00:00<00:00, 106MB/s]
 61%|██████▏   | 56.6M/92.2M [00:00<00:00, 117MB/s]
 80%|███████▉  | 73.4M/92.2M [00:00<00:00, 133MB/s]
100%|██████████| 92.2M/92.2M [00:00<00:00, 120MB/s]
  Extracting: 100%|██████████| 60/60 [00:05<00:00, 10.54it/s]



⏳ Processing DANG NGUYEN PHUONG NGHI (60 images)...


...


⏳ Processing TRAN THE DANG KHOA (60 images)...


  Extracting: 100%|██████████| 60/60 [00:00<00:00, 2543.93it/s]



⏳ Processing TRINH THUY HANG (60 images)...


  Extracting: 100%|██████████| 60/60 [00:00<00:00, 2587.13it/s]


✅ Valid embeddings: 0 | Skipped (lỗi/không có mặt): 1925
📊 Shape: (0,)


In [ ]:
# Test với 1 ảnh đầu tiên của người đầu tiên
import os, cv2
from deepface import DeepFace

person = class_folders[0]
img_file = [f for f in os.listdir(os.path.join(DATA_PATH, person)) if f.lower().endswith(('.jpg','.png'))][0]
img_path = os.path.join(DATA_PATH, person, img_file)

print(f"🔍 Testing: {img_path}")
print(f"📁 File exists: {os.path.exists(img_path)}")

try:
    # Thử với enforce_detection=False để xem có extract được không
    result = DeepFace.represent(img_path, model_name='Facenet', enforce_detection=False)
    print(f"✅ Success! Embedding shape: {len(result[0]['embedding']) if result else 'No result'}")
except Exception as e:
    print(f"❌ Error: {type(e).__name__}: {str(e)[:200]}")

🔍 Testing: /content/drive/MyDrive/60 ẢNH/BUI DANG KHOI/z7850577423905_8d96a5b55c870baa4b38f22d516c124d.jpg
📁 File exists: True
❌ Error: ValueError: Input image must not have non-english characters - /content/drive/MyDrive/60 ẢNH/BUI DANG KHOI/z7850577423905_8d96a5b55c870baa4b38f22d516c124d.jpg


In [ ]:
import os, shutil
from tqdm import tqdm

SRC_PATH = '/content/drive/MyDrive/60 ẢNH'
DEST_PATH = '/content/dataset_clean'  # Path không dấu, DeepFace chấp nhận

# Tạo folder đích
os.makedirs(DEST_PATH, exist_ok=True)

# Copy toàn bộ cấu trúc folder + ảnh sang path mới
print(f"📦 Copying dataset from {SRC_PATH} → {DEST_PATH}...")
for person_folder in os.listdir(SRC_PATH):
    src_person = os.path.join(SRC_PATH, person_folder)
    dest_person = os.path.join(DEST_PATH, person_folder)

    if os.path.isdir(src_person):
        os.makedirs(dest_person, exist_ok=True)
        img_files = [f for f in os.listdir(src_person) if f.lower().endswith(('.jpg','.jpeg','.png'))]

        for img in tqdm(img_files, desc=f"Copying {person_folder}"):
            shutil.copy2(
                os.path.join(src_person, img),
                os.path.join(dest_person, img)
            )

print(f"✅ Done! Dataset sạch đã ở: {DEST_PATH}")
print(f"📁 Kiểm tra nhanh: {os.listdir(DEST_PATH)[:3]}")

📦 Copying dataset from /content/drive/MyDrive/60 ẢNH → /content/dataset_clean...


Copying HOANG BUI TRA MY:  75%|███████▌  | 45/60 [00:15<00:04,  3.14it/s]

In [ ]:
# === CELL 3 - FIX VERSION (DÙNG PATH ASCII) ===
from deepface import DeepFace
import os, numpy as np, joblib
from tqdm import tqdm

# 👉 DÙNG PATH SẠCH KHÔNG DẤU
DATA_PATH = '/content/dataset_clean'  # <-- THAY ĐỔI QUAN TRỌNG

class_folders = sorted([f for f in os.listdir(DATA_PATH) if os.path.isdir(os.path.join(DATA_PATH, f))])
print(f"📁 Processing {len(class_folders)} classes from {DATA_PATH}")

X, y, skipped = [], [], 0

for idx, person_name in enumerate(class_folders):
    person_path = os.path.join(DATA_PATH, person_name)
    img_files = [f for f in os.listdir(person_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    for img_file in tqdm(img_files, desc=f"{person_name}"):
        try:
            img_path = os.path.join(person_path, img_file)

            # FIX: enforce_detection=False + fallback model
            try:
                objs = DeepFace.represent(img_path, model_name='Facenet', enforce_detection=False)
            except:
                objs = DeepFace.represent(img_path, model_name='VGG-Face', enforce_detection=False)

            if objs and len(objs) > 0 and "embedding" in objs[0]:
                emb = objs[0]["embedding"]
                # Kiểm tra embedding hợp lệ
                if np.sum(np.abs(emb)) > 1e-6:
                    X.append(emb)
                    y.append(idx)
                else:
                    skipped += 1
            else:
                skipped += 1

        except Exception:
            skipped += 1
            continue

X = np.array(X) if X else np.empty((0, 128))
y = np.array(y)

print(f"\n✅ Valid embeddings: {len(X)} | Skipped: {skipped}")
if len(X) > 0:
    print(f"📊 Shape: {X.shape} | Success rate: {len(X)/(len(X)+skipped)*100:.1f}%")
else:
    print("❌ Vẫn 0 embedding? Kiểm tra lại ảnh mẫu!")

# Save
np.save('/content/X.npy', X)
np.save('/content/y.npy', y)

📁 Processing 31 classes from /content/dataset_clean


NGUYEN NGOC KIM TUYET:   0%|          | 0/60 [00:00<?, ?it/s]

26-05-28 12:58:13 - 🔗 vgg_face_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/vgg_face_weights.h5 to /root/.deepface/weights/vgg_face_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/vgg_face_weights.h5
To: /root/.deepface/weights/vgg_face_weights.h5

  0%|          | 0.00/580M [00:00<?, ?B/s]
  2%|▏         | 11.0M/580M [00:00<00:06, 89.0MB/s]
  4%|▍         | 23.6M/580M [00:00<00:05, 109MB/s] 
  7%|▋         | 37.7M/580M [00:00<00:04, 123MB/s]
  9%|▊         | 50.3M/580M [00:00<00:04, 124MB/s]
 11%|█         | 63.4M/580M [00:00<00:04, 112MB/s]
 13%|█▎        | 76.5M/580M [00:00<00:04, 117MB/s]
 15%|█▌        | 89.1M/580M [00:00<00:04, 119MB/s]
 18%|█▊        | 102M/580M [00:00<00:04, 111MB/s] 
 20%|█▉        | 113M/580M [00:01<00:04, 107MB/s]
 21%|██▏       | 124M/580M [00:01<00:04, 101MB/s]
 23%|██▎       | 136M/580M [00:01<00:04, 106MB/s]
 25%|██▌       | 147M/580M [00:01<00:04, 102MB/s]
 27%|██▋       | 158M/580M [00:01<00:04, 103MB/s]
 29%|██▉       | 170M/580M [00:01<00:03, 108MB/s]
 31%|███▏      | 182M/580M [00:01<00:03, 111MB/s]
 34%|███▎      | 195M/580M [00:01<00:03,


✅ Valid embeddings: 1865 | Skipped: 60
📊 Shape: (1865, 128) | Success rate: 96.9%


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
import joblib
import numpy as np

# Load data đã extract
X = np.load('/content/X.npy')
y = np.load('/content/y.npy')

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"📊 Train: {len(X_train)} | Test: {len(X_test)}")

# Train ANN (MLP)
print("\n⏳ Training ANN...")
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),  # 2 lớp ẩn = ANN chuẩn
    activation='relu',
    alpha=0.001,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.15,
    verbose=True
)

mlp.fit(X_train, y_train)

# Eval
train_acc = mlp.score(X_train, y_train)
test_acc = mlp.score(X_test, y_test)

print(f"\n✅ Training done!")
print(f"📈 Train accuracy: {train_acc*100:.2f}%")
print(f"📈 Test accuracy: {test_acc*100:.2f}%")

# Save model
joblib.dump(mlp, '/content/face_ann.pkl')
joblib.dump(class_folders, '/content/class_names.pkl')
print("💾 Model saved!")

📊 Train: 1492 | Test: 373

⏳ Training ANN...
Iteration 1, loss = 3.39608965
Validation score: 0.147321
Iteration 2, loss = 3.03528986
Validation score: 0.236607
Iteration 3, loss = 2.74699969
Validation score: 0.348214
Iteration 4, loss = 2.43349536
Validation score: 0.437500
Iteration 5, loss = 2.11048843
Validation score: 0.500000
Iteration 6, loss = 1.81383229
Validation score: 0.553571
Iteration 7, loss = 1.56606886
Validation score: 0.571429
Iteration 8, loss = 1.37682101
Validation score: 0.607143
Iteration 9, loss = 1.23528820
Validation score: 0.625000
Iteration 10, loss = 1.13079366
Validation score: 0.633929
Iteration 11, loss = 1.04816874
Validation score: 0.647321
Iteration 12, loss = 0.97848982
Validation score: 0.669643
Iteration 13, loss = 0.92340731
Validation score: 0.669643
Iteration 14, loss = 0.87773049
Validation score: 0.665179
Iteration 15, loss = 0.84227799
Validation score: 0.683036
Iteration 16, loss = 0.80490397
Validation score: 0.683036
Iteration 17, loss =

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np, joblib

# Load data
X = np.load('/content/X.npy')
y = np.load('/content/y.npy')

# One-hot encode labels cho 31 classes
from tensorflow.keras.utils import to_categorical
y_cat = to_categorical(y, num_classes=len(class_folders))

# Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_cat, test_size=0.2, random_state=42, stratify=y
)

# Build model: ANN thuần (Dense layers)
model = keras.Sequential([
    layers.Input(shape=(128,)),  # Input = embedding 128-d từ CNN
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(class_folders), activation='softmax')  # Output 31 classes
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\n🚀 Training with Keras:")
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1  # <--- Đây là dòng tạo progress bar đẹp!
)

# Eval
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\n✅ Test accuracy: {test_acc*100:.2f}%")

# Save
model.save('/content/face_ann_keras.h5')
joblib.dump(class_folders, '/content/class_names.pkl')
print("💾 Saved!")


🚀 Training with Keras (có progress bar epoch):
Epoch 1/30
47/47 [==============================] - 7s 11ms/step - loss: 3.3205 - accuracy: 0.0925 - val_loss: 2.9464 - val_accuracy: 0.3539
Epoch 2/30
47/47 [==============================] - 0s 5ms/step - loss: 2.7495 - accuracy: 0.2567 - val_loss: 2.2310 - val_accuracy: 0.5308
Epoch 3/30
47/47 [==============================] - 0s 5ms/step - loss: 2.1869 - accuracy: 0.3975 - val_loss: 1.7120 - val_accuracy: 0.6273
Epoch 4/30
47/47 [==============================] - 0s 5ms/step - loss: 1.8364 - accuracy: 0.5027 - val_loss: 1.4533 - val_accuracy: 0.6515
Epoch 5/30
47/47 [==============================] - 0s 5ms/step - loss: 1.6105 - accuracy: 0.5476 - val_loss: 1.3379 - val_accuracy: 0.6783
Epoch 6/30
47/47 [==============================] - 0s 5ms/step - loss: 1.4744 - accuracy: 0.6025 - val_loss: 1.2617 - val_accuracy: 0.6890
Epoch 7/30
47/47 [==============================] - 0s 5ms/step - loss: 1.3554 - accuracy: 0.6260 - val_loss: 1

In [ ]:
import os, cv2, numpy as np, tensorflow as tf, joblib, random
from deepface import DeepFace

# Load model & data
model = tf.keras.models.load_model('/content/face_ann_keras.h5')
class_names = joblib.load('/content/class_names.pkl')
DATA_PATH = '/content/dataset_clean'

def recognize_safe(img_path):
    """Bypass lỗi path tiếng Việt bằng cách copy sang temp path ASCII"""
    temp_path = '/content/temp_face.jpg'
    img = cv2.imread(img_path)
    if img is None: return "❌ Lỗi đọc ảnh", 0.0
    cv2.imwrite(temp_path, img)  # Lưu sang path sạch

    try:
        objs = DeepFace.represent(temp_path, model_name='Facenet', enforce_detection=False)
        if not objs or "embedding" not in objs[0]:
            return "❌ Không detect được mặt", 0.0

        emb = np.array(objs[0]["embedding"]).reshape(1, -1)
        probs = model.predict(emb, verbose=0)[0]
        pred_idx = np.argmax(probs)
        confidence = probs[pred_idx] * 100
        return class_names[pred_idx], confidence
    except Exception as e:
        return f"️ Lỗi: {str(e)[:25]}", 0.0

print("🧪 LIVE DEMO — Nhận diện khuôn mặt\n" + "="*50)
test_people = random.sample(class_names, 5)
correct = 0
tested = 0

for name in test_people:
    folder = os.path.join(DATA_PATH, name)
    imgs = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg','.png'))]
    if not imgs: continue

    # Thử tối đa 2 ảnh/người để tránh ảnh góc xấu/filter nặng
    success = False
    for _ in range(2):
        test_img = os.path.join(folder, random.choice(imgs))
        pred, conf = recognize_safe(test_img)

        if "Lỗi" in pred or "Không detect" in pred:
            continue  # Ảnh lỗi, thử ảnh khác

        tested += 1
        is_correct = (pred == name)
        if is_correct: correct += 1
        status = "✅" if is_correct else "❌"
        print(f"{status} Ground truth: {name}")
        print(f"   → Predicted: {pred} | Confidence: {conf:.1f}%\n")
        success = True
        break  # Sang người tiếp theo

    if not success:
        print(f"⚠️ {name}: Không có ảnh hợp lệ để test\n")

print(f"📊 Kết quả demo: {correct}/{tested} đúng")
print("💡 Lưu ý: Dataset thực tế có filter/góc nghiêng đa dạng → 60-75% là kết quả thực tế tốt.")

🧪 LIVE DEMO — Nhận diện khuôn mặt


FileNotFoundError: [Errno 2] No such file or directory: '/content/dataset_clean/PHAM DUC THANH CONG'

In [17]:
from google.colab import files
import cv2, os, numpy as np, tensorflow as tf, joblib
from deepface import DeepFace

# Load model
model = tf.keras.models.load_model('/content/face_ann_keras.h5')
class_names = joblib.load('/content/class_names.pkl')

def recognize_from_path(img_path):
    """Nhận diện ảnh từ đường dẫn"""
    temp_path = '/content/temp_test.jpg'
    img = cv2.imread(img_path)
    if img is None:
        return "❌ Không đọc được ảnh", 0.0
    cv2.imwrite(temp_path, img)

    try:
        objs = DeepFace.represent(temp_path, model_name='Facenet', enforce_detection=False)
        if not objs or "embedding" not in objs[0]:
            return "❌ Không detect được khuôn mặt", 0.0

        emb = np.array(objs[0]["embedding"]).reshape(1, -1)
        probs = model.predict(emb, verbose=0)[0]
        pred_idx = np.argmax(probs)
        confidence = probs[pred_idx] * 100

        # Top 3 dự đoán
        top_3_idx = np.argsort(probs)[::-1][:3]
        print("📊 Top 3 dự đoán:")
        for i, idx in enumerate(top_3_idx, 1):
            print(f"   {i}. {class_names[idx]} ({probs[idx]*100:.1f}%)")

        return class_names[pred_idx], confidence
    except Exception as e:
        return f"⚠️ Lỗi: {str(e)[:30]}", 0.0

# 🎯 UPLOAD ẢNH TỪ MÁY
print("📤 Upload ảnh từ máy tính của bạn...")
print("💡 Chọn 1 ảnh chứa khuôn mặt (JPG/PNG)\n")

uploaded = files.upload()

if uploaded:
    img_filename = list(uploaded.keys())[0]
    print(f"\n✅ Đã upload: {img_filename}")
    print("="*50)

    # Nhận diện
    pred_name, confidence = recognize_from_path(img_filename)

    print("\n" + "="*50)
    print("🎯 KẾT QUẢ NHẬN DIỆN:")
    print(f"   → Tên: {pred_name}")
    print(f"   → Confidence: {confidence:.1f}%")
    print("="*50)

    if confidence >= 70:
        print("✅ Model tự tin với kết quả này!")
    elif confidence >= 50:
        print("⚠️ Model hơi phân vân, có thể kiểm tra lại.")
    else:
        print("❌ Model không chắc chắn - có thể là người ngoài lớp hoặc ảnh chất lượng thấp.")

📤 Upload ảnh từ máy tính của bạn...
💡 Chọn 1 ảnh chứa khuôn mặt (JPG/PNG)



Saving NTM_test.png to NTM_test (1).png

✅ Đã upload: NTM_test (1).png
📊 Top 3 dự đoán:
   1. NGUYEN TRONG MINH (99.8%)
   2. NGUYEN HUU TOAN (0.2%)
   3. THAI TUAN PHAT (0.0%)

🎯 KẾT QUẢ NHẬN DIỆN:
   → Tên: NGUYEN TRONG MINH
   → Confidence: 99.8%
✅ Model tự tin với kết quả này!


In [ ]:
# === CELL PHỤC HỒI ===
import os, cv2, numpy as np, tensorflow as tf, joblib
from google.colab import drive, files
from deepface import DeepFace

# Check xem model còn không
if os.path.exists('/content/face_ann_keras.h5'):
    print("✅ Model vẫn còn trong /content!")
    model = tf.keras.models.load_model('/content/face_ann_keras.h5')
    class_names = joblib.load('/content/class_names.pkl')
    print(f"📂 Loaded {len(class_names)} classes")

else:
    print("❌ Model đã mất (do disconnect lâu)")
    print("📥 Download từ Drive hoặc re-train nhanh...")

    # Mount Drive để lấy model đã lưu
    drive.mount('/content/drive', force_remount=True)

    # Tìm file model trong Drive
    import subprocess
    result = subprocess.run(['find', '/content/drive', '-name', 'face_ann_keras.h5'],
                          capture_output=True, text=True)

    if result.stdout.strip():
        model_path = result.stdout.strip().split('\n')[0]
        print(f"✅ Found model: {model_path}")

        # Copy về /content
        import shutil
        shutil.copy(model_path, '/content/face_ann_keras.h5')
        shutil.copy('/content/drive/MyDrive/class_names.pkl', '/content/class_names.pkl')

        model = tf.keras.models.load_model('/content/face_ann_keras.h5')
        class_names = joblib.load('/content/class_names.pkl')
        print("✅ Model loaded successfully!")
    else:
        print("⚠️ Không tìm thấy model trong Drive. Cần re-train từ đầu.")
        print("👉 Chạy lại Cell 3-4 từ đầu (embedding + train)")

print("\n" + "="*50)
print("✅ PHỤC HỒI HOÀN TẤT - SẴN SÀNG TEST!")
print("="*50)

ModuleNotFoundError: No module named 'deepface'

In [ ]:
import os, shutil, datetime
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive', force_remount=True)

# 2. Tạo thư mục backup có timestamp
backup_dir = f"/content/drive/MyDrive/FaceRec_Class31_{datetime.datetime.now().strftime('%Y-%m-%d')}"
os.makedirs(backup_dir, exist_ok=True)

# 3. Copy các file quan trọng
files_to_save = [
    '/content/face_ann_keras.h5',
    '/content/class_names.pkl',
    '/content/X.npy',
    '/content/y.npy'
]

print(f" Đang lưu vào: {backup_dir}")
for f in files_to_save:
    if os.path.exists(f):
        shutil.copy(f, backup_dir)
        print(f"  ✅ {os.path.basename(f)}")
    else:
        print(f"  ⚠️ Không tìm thấy {f}")

# 4. Lưu cả notebook hiện tại (đề phòng)
import json
notebook_path = '/content/Untitled0.ipynb'  # Thay tên nếu notebook của bạn khác
if os.path.exists(notebook_path):
    shutil.copy(notebook_path, backup_dir)
    print("  ✅ Notebook.ipynb")

print("\n🎉 HOÀN TẤT! Bạn có thể TẮT MÁY an toàn.")
print("📂 Đường dẫn backup:", backup_dir)

Mounted at /content/drive
 Đang lưu vào: /content/drive/MyDrive/FaceRec_Class31_2026-05-28
  ✅ face_ann_keras.h5
  ✅ class_names.pkl
  ✅ X.npy
  ✅ y.npy

🎉 HOÀN TẤT! Bạn có thể TẮT MÁY an toàn.
📂 Đường dẫn backup: /content/drive/MyDrive/FaceRec_Class31_2026-05-28


In [3]:
# === CELL KHÔI PHỤC ===
import os, joblib, numpy as np
import tensorflow as tf
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive', force_remount=True)

# 2. Tìm thư mục backup (tự động lấy thư mục mới nhất)
backup_base = '/content/drive/MyDrive'
folders = [f for f in os.listdir(backup_base) if f.startswith('FaceRec_Class31_')]
folders.sort(reverse=True)  # Lấy bản mới nhất
backup_dir = os.path.join(backup_base, folders[0])

print(f"📂 Đang khôi phục từ: {backup_dir}")

# 3. Copy file về /content để Colab dùng được
import shutil
for f in ['face_ann_keras.h5', 'class_names.pkl']:
    src = os.path.join(backup_dir, f)
    dst = f'/content/{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"  ✅ Đã khôi phục {f}")

# 4. Load model & class names vào memory
model = tf.keras.models.load_model('/content/face_ann_keras.h5')
class_names = joblib.load('/content/class_names.pkl')

print(f"\n🚀 SN SÀNG! Đã load {len(class_names)} classes.")
print(" Giờ chạy cell Test/Demo như bình thường.")

Mounted at /content/drive
📂 Đang khôi phục từ: /content/drive/MyDrive/FaceRec_Class31_2026-05-28
  ✅ Đã khôi phục face_ann_keras.h5
  ✅ Đã khôi phục class_names.pkl



🚀 SN SÀNG! Đã load 31 classes.
 Giờ chạy cell Test/Demo như bình thường.
